In [2]:
!git clone https://github.com/eliftufekci/k_shortest_path_with_diversity.git

fatal: destination path 'k_shortest_path_with_diversity' already exists and is not an empty directory.


In [3]:
import sys

top_level_container_dir = "k_shortest_path_with_diversity/"

if top_level_container_dir not in sys.path:
    sys.path.insert(0, top_level_container_dir)

In [ ]:
import gc
import networkx as nx
import random
import datetime
import numpy as np

from k_shortest_path_with_diversity.examples import draw_bar_chart, download_and_prepare_graphs
from k_shortest_path_with_diversity.examples.draw_distribution import draw_time_distribution, draw_num_of_path_distribution
from k_shortest_path_with_diversity.src.algorithms import FindKSP, FindIterBound

In [ ]:
def run_algorithm(algorithm, G, threshold, k, node_pairs):
    times = []
    num_paths = []

    for src, dest in node_pairs:
        print(f"\nComparing algorithms for SRC: {src}, DEST: {dest}")

        start_time = datetime.datetime.now()
        alg = algorithm(G, threshold)
        result = alg.find_paths(src=src, dest=dest, k=k)
        end_time = datetime.datetime.now()
        execution_time = end_time - start_time

        times.append(execution_time.total_seconds())
        num_paths.append(alg.number_of_paths_explored)

    return times, num_paths

In [6]:
def upload_graph(filepath, num_pairs):
    G = nx.DiGraph()
    with open(filepath) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 3:  # Weighted graph (3 parçalı satır: kaynak, hedef, ağırlık)
                u, v, weight = int(parts[0]), int(parts[1]), float(parts[2])
                G.add_edge(u, v, weight=weight)
            elif len(parts) == 2:  # Unweighted graph (2 parçalı satır: kaynak ve hedef)
                u, v = map(int, parts)
                G.add_edge(u, v, weight=1)  # Ağırlıksız kenarları varsayılan olarak '1' ağırlıkla ekle
            else:
                raise ValueError("Unexpected line format in graph file.")

    node_pairs = []
    for _ in range(num_pairs):
        src = random.choice(list(G.nodes()))
        reachable = list(nx.descendants(G, src))

        while not reachable:
            src = random.choice(list(G.nodes()))
            reachable = list(nx.descendants(G, src))

        dest = random.choice(reachable)
        node_pairs.append((src, dest))

    return G, node_pairs

In [ ]:
download_and_prepare_graphs()

k_list = [10, 20, 30, 40, 50]
diversity_threshold = 0.6 #not important #not important

Downloading: https://snap.stanford.edu/data/web-Google.txt.gz
Extracting: /content/graph-data/web-Google.txt.gz -> /content/graph-data/web-Google.txt
Removed archive: /content/graph-data/web-Google.txt.gz
Done: /content/graph-data/web-Google.txt
Downloading: https://snap.stanford.edu/data/wiki-Talk.txt.gz
Extracting: /content/graph-data/wiki-Talk.txt.gz -> /content/graph-data/wiki-Talk.txt
Removed archive: /content/graph-data/wiki-Talk.txt.gz
Done: /content/graph-data/wiki-Talk.txt
Downloading: https://www.diag.uniroma1.it/challenge9/data/USA-road-d/USA-road-d.FLA.gr.gz
Extracting: /content/graph-data/USA-road-d.FLA.gr.gz -> /content/graph-data/USA-road-d.FLA.gr
Removed archive: /content/graph-data/USA-road-d.FLA.gr.gz
Done: /content/graph-data/USA-road-d.FLA.gr
Downloading: https://www.diag.uniroma1.it/challenge9/data/USA-road-d/USA-road-d.COL.gr.gz
Extracting: /content/graph-data/USA-road-d.COL.gr.gz -> /content/graph-data/USA-road-d.COL.gr
Removed archive: /content/graph-data/USA-ro

In [ ]:
roadFLA_path = "/content/graph-data/USA-road-d.FLA.gr"
num_pairs = 5
G, node_pairs = upload_graph(roadFLA_path, num_pairs)
print(node_pairs)

In [ ]:
all_results = []

for k_to_find in k_list:
    print("working with KSP")
    ksp_times, ksp_num_paths = run_algorithm(FindKSP, G, diversity_threshold, k_to_find, node_pairs)
    print(f"ksp_times= {ksp_times}")
    print(f"ksp_num_paths= {ksp_num_paths}")
    print("working with Iterbound")
    iterbound_times, iterbound_num_paths = run_algorithm(FindIterBound, G, diversity_threshold, k_to_find, node_pairs)
    print(f"iterbound_times= {iterbound_times}")
    print(f"iterbound_num_paths= {iterbound_num_paths}")

    roadFLA_result = (
        (np.average(ksp_times) if ksp_times else 0,
         np.average(ksp_num_paths) if ksp_num_paths else 0,
         ksp_times, ksp_num_paths),
        (np.average(iterbound_times) if iterbound_times else 0,
         np.average(iterbound_num_paths) if iterbound_num_paths else 0,
         iterbound_times, iterbound_num_paths)
    )

    print(roadFLA_result)
    all_results.append(roadFLA_result)

In [ ]:
graph_types = ("RoadFLA",)

algorithms_paths = {
    'FindKSP':       [r[0][1] for r in all_results],  # index 1 = avg_num_paths
    'FindIterbound': [r[1][1] for r in all_results],
}

algorithms_time = {
    'FindKSP':       [r[0][0] for r in all_results],  # index 0 = avg_time
    'FindIterbound': [r[1][0] for r in all_results],
}

markers = {
    'FindKSP':       's',  # □ kare
    'FindIterbound': '^',  # △ üçgen
}

draw_line_chart(k_list, markers, algorithms_paths, algorithms_time, graph_name="RoadFLA")

In [ ]:
all_ksp_times = []
all_iterbound_times = []
all_ksp_num_paths = []
all_iterbound_num_paths = []

for result in all_results:
    all_ksp_times.extend(result[0][2])
    all_ksp_num_paths.extend(result[0][3])
    all_iterbound_times.extend(result[1][2])
    all_iterbound_num_paths.extend(result[1][3])

# Plotting distributions for times
plot_configs_time = [
    ('FindKSP Execution Times', all_ksp_times, 'skyblue'),
    ('FindIterBound Execution Times', all_iterbound_times, 'lightcoral'),
]
draw_time_distribution(plot_configs_time)

# Plotting distributions for number of paths
plot_configs_num_paths = [
    ('FindKSP Number of Paths Explored', all_ksp_num_paths, 'skyblue'),
    ('FindIterBound Number of Paths Explored', all_iterbound_num_paths, 'lightcoral'),
]
draw_num_of_path_distribution(plot_configs_num_paths)